## BASE DE DADOS

Base baixada do site https://www.consumerfinance.gov/data-research/consumer-complaints/search/?date_received_max=2026-03-19&date_received_min=2024-07-01&has_narrative=true&page=1&searchField=all&size=25&sort=created_date_desc&tab=List

Nesse site encontramos uma base de dados com reclamações e demandas do setor financeiro americano.

Os dados utilzados nesse projeto foram filtrados dessa base, da data de 01/07/2024 a 19/03/2026, selecionando apenasos registros que possuíam narrativa.

## Configurações básicas do notebook

In [3]:
from dataclasses import dataclass
# Aqui configuramos os principais parâmetros da pipeline
@dataclass
class Config:
    input_csv: str = "complaints-2026-02-19_18_26.csv" #nome do arquivo de entrada
    output_csv: str = "resultado_460000.csv" #nome do arquivo de saída
    chunksize: int = 10000 #tamanho do chunk para leitura do CSV
    random_state: int = 42
    roberta_model: str = "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis" #modelo Roberta
    os_model: str = "all-MiniLM-L6-v2" #modelo de embeddings
    max_len_chars: int = 512 #tamanho máximo de caracteres para o modelo distilroberta-finetuned
    #hf_token_env: str = # se existir, usa para autenticar no hub
    wanted_cols = ['Product', 'Consumer complaint narrative'] #colunas utilizadas para o modelo
    sample: bool = False #se for rodar em amostra
    sample_size: int = 1000   #tamanho da amostra, se sample for True

## Importando bibliotecas

In [5]:
# AO FINAL, LEMBRAR DE GERAR O ARQUIVO TXT: pip freeze > requirements.txt
'''
Para rodar a pipeline, na primeira vez, é necessário estar conectado na Internet, pois baixamos
os dados das bibliotecas de NLP.

Para instalar as bibliotecas necessárias execute o comando abaixo no terminal:
pip install -r requirements.txt

Abaixo importamos as bibliotecas necessárias
'''
import os
import re
import logging
import string
import unicodedata
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd

# Sklearn (para n-gramas opcionais)
from sklearn.feature_extraction.text import CountVectorizer

# NLTK (VADER + preprocess)
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Transformers (DistilRoBERTa)
from transformers import pipeline

# Sentence-Transformers (embeddings)
from sentence_transformers import SentenceTransformer

#os.environ["HF_TOKEN"] = Config.hf_token_env  # Configura o token do Hugging Face, se existir

## Funções de leitura de dados e Log

In [ ]:
# Configuração de um sistema básico de Log    
def setup_logging() -> None:
    fmt = "%(asctime)s | %(levelname)s | %(message)s"
    logging.basicConfig(level=logging.INFO, format=fmt)
    
'''
Aqui reunimos as classe de Processamento de Texto e Leitura dos dados
do CSV, montando um DataFrame para utilizar na pipeline

DataLoader.load -> Lê a base de dados em CSV e faz o tratamento básico.

TextPreprocessor.normalize_str -> utilizado para a coluna de texto processado para o Roberta e OS
TextPreprocessor.pre_processamento_vader -> utilizado para o texto processado para o Vader
TextPreprocessor.lematizador -> Divide texto em tokens, remove stopwords e lematiza o texto. Utilizar para o treinamento do modelo de ML
'''

# ============================
# Leitura de dados
# ============================
class DataLoader:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load(self) -> pd.DataFrame:
        wanted_cols = self.cfg.wanted_cols
        array_df = []
        primeira = 0
        for chunk in pd.read_csv(self.cfg.input_csv, sep=',', chunksize=self.cfg.chunksize):
            if primeira == 0:
                primeira = 1
                missing = [c for c in wanted_cols if c not in chunk.columns]
                if missing:
                    raise ValueError(f"Colunas ausentes no CSV: {missing}")
            temp_df = chunk[wanted_cols]
            array_df.append(temp_df)
        df = pd.concat(array_df, ignore_index=True)
        # tirar relatos repetidos (base possui muitos)
        df = df.drop_duplicates().reset_index(drop=True)
        # Garante string
        for c in wanted_cols:
            df[c] = df[c].astype(str)
        # Renomeia colunas para facilitar trabalho
        df.rename(columns={'Consumer complaint narrative': 'narrative', 'Product': 'product'}, inplace=True)
        logging.info("Dataset final: %s linhas | colunas=%s", len(df), list(df.columns))
        return df



## Classe de tratamento dos dados textuais

In [11]:
# ============================
# Pré-processamento de texto
# ============================
class TextPreprocessor:
    
    LISTA_SIMBOLOS = ['(', ')', '@', '"', '+', '-', '.', ',', '\\', '/']

    @staticmethod
    def normalize_accents(text: str) -> str:
        return unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("utf-8")

    @staticmethod
    def remove_punctuation(text: str) -> str:
        table = str.maketrans({key: " " for key in string.punctuation})
        return text.translate(table)

    @staticmethod
    def remove_xxxx(text: str) -> str:
        text = re.sub(r"x{2,}", " ", text)
        text = re.sub(r"X{2,}", " ", text)
        return text

    @classmethod
    def normalize_str(cls, text: str) -> str:
        # lower, remover pontuação, acentos, sequências de X e compressão de espaços
        # esse processamento servirá para o modelo de orientação semâmtica e para o
        # Roberta
        t = text.lower()
        t = cls.remove_punctuation(t)
        t = cls.normalize_accents(t)
        t = cls.remove_xxxx(t)
        t = re.sub(re.compile(r" +"), " ", t)
        return " ".join([w for w in t.split()])

    @classmethod
    def pre_processamento_vader(cls, text: str) -> str:
        """Pré-processamento específico p/ VADER (não lematiza, mantém *stopwords* e *case*)."""
        # mantém caixa (sensível no VADER), remove apenas símbolos listados e sequências 'xxxx'
        t = re.sub(r"\s+", " ", text)
        t = cls.remove_xxxx(t)
        tokens = word_tokenize(t)  # mantém pontuação exceto os símbolos que retiraremos
        tokens = [tok for tok in tokens if tok not in cls.LISTA_SIMBOLOS]
        return " ".join(tokens)
    
# ========================================================================================
# Pré-processamento de texto para o modelo de ML (lematização, remoção de stopwords, etc)
# ========================================================================================

    @classmethod
    def lematizador(cls, text: str) -> str:
        '''
        Recebe uma string e tokeniza ela. 
        Retira as stopwords
        Aplica a WordNetLemmatizer() a cada token.
        Devolve uma string após o join() da lista.
        '''
        stop_words = nltk.corpus.stopwords.words("english") # portuguese, caso o dataset seja em português
        if isinstance(text, str):
            text = cls.normalize_str(text) #aplica a função acima
            text = word_tokenize(text) #tokeniza
            text = [x for x in text if x not in stop_words] #remove stopwords
            text = [y for y in text if len(y) > 2] #remove letras isoladas
            sentence = []
            #aplica lematizador em cada um dos tokens que sejam verbos
            for word in text:
                lemmatizer = WordNetLemmatizer()
                sentence.append(lemmatizer.lemmatize(word,'v'))
            return ' '.join(sentence) #devolve uma string unica
        else:
            return None

## Modelos de predição das classes.

Esse modelos realizam a predição preliminar da polaridade dos textos. Utilizei três modelos diferentes, e ao final atribuí um peso para cada modelo, considerando a eficácia deles a partir de análise manual de alguns casos de classificação aleatórios, dentro da base.

Logo após, utilizamos esse pesos para criar uma escala, condensando o resultado obtido por cada modelo, criando logo após uma label preliminar. Esta tabela, com os textos e a label final será utilizada para o treinamento do modelo de ML especializado.

### Sobre cada modelos:
#### Vader
#### DistilroBERTa (Roberta)
#### Orientação Semântica (MiniLM utilizado para calcular Embeddings)

In [ ]:
# ============================
# VADER
# ============================
class VaderAnalyzer:
    def __init__(self):
        self._ensure_nltk()
        self.sid = SentimentIntensityAnalyzer()

    @staticmethod
    def _ensure_nltk():
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            nltk.download('punkt', quiet=True)
        try:
            nltk.data.find('sentiment/vader_lexicon')
        except LookupError:
            nltk.download('vader_lexicon', quiet=True)

    @staticmethod
    def label_from_compound(val: float) -> str:
        # Mesma lógica do notebook
        if val > 0.5:
            return 'positive'
        elif val < -0.5:
            return 'very negative'
        elif val <= 0 and val >= -0.5:
            return 'negative'
        else:
            return 'neutral'

    def score(self, text_for_vader: str) -> float:
        scores = self.sid.polarity_scores(text_for_vader)
        return scores['compound']


# ============================
# DistilRoBERTa (HF pipeline)
# ============================
class RobertaAnalyzer:
    def __init__(self, cfg: Config, details=False):
        self.details = details
        self.cfg = cfg
        self.pipe = pipeline("text-classification", model= self.cfg.roberta_model, hf_token=self.cfg.hf_token_env)

    # Para lidar com textos longos, dividimos em pedaços e depois agregamos os resultados    
    def aplica_roberta(self, texto):
        max_lenght = self.cfg.max_len_chars
        scores = []
        n = len(texto)
        for start in range(0, n, max_lenght):
            end = min(start + max_lenght, n)
            batch = texto[start:end]
            resultado = self.pipe(batch)
            scores.append(resultado)
        labels = [i[0]['label'] for i in scores]
        values = [i[0]['score'] for i in scores]
        resultado_final = (values,labels)
        label = self.get_roberta_def_label(labels, values)
        if self.details:
            return resultado_final
        else:
            return label

    # Agrega os resultados dos pedaços para definir a label final do texto completo
    # Aqui a lógica é simples: somamos os scores de cada label e a que tiver o maior score total é a label final do texto completo
    @staticmethod
    def get_roberta_def_label(labels, valores):
        pos, neu, neg = 0, 0, 0
        for label, valor in zip(labels, valores):
            if label == 'positive':
                pos += valor
            if label == 'neutral':
                neu += valor
            if label == 'negative':
                neg += valor
        if (pos > neu) & (pos > neg):
            return 'positive'
        if (neu > pos) & (neu > neg):
            return 'neutral'
        else:
            return 'negative'

# ============================
# Embeddings + seeds (MiniLM)
# ============================
# Para o modelo de orientação semântica, utilizamos o modelo de embeddings "all-MiniLM-L6-v2" da Sentence-Transformers, que é leve e eficiente.
# A ideia é calcular a média da similaridade do texto com um conjunto de seeds positivas, negativas e muito negativas, e assim definir a label final.
class EmbeddingSentiment:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)
        # Seeds melhoradas (como na última versão do notebook)
        self.seeds_pos = [
            "resolved", "helpful", "excellent", "satisfied", "fast response",
            "issue fixed", "reliable", "transparent", "approved", "appreciation", "successfully",
            "positive experience", "good service", "recommend", "thank you", "trustworthy"
        ]
        self.seeds_neg = [
            "delayed", "unresponsive", "blocked", "denied", "long waiting time", "poor service",
            "assistance", "inaccurate", "without reporting", "does not match", "systematic error", "not provided"
        ]
        self.seeds_very_neg = [
            "fraud", "unacceptable", "hidden fees", "scam", "unauthorized",
            "chargeback", "violation", "identity theft", "damage to reputation", "illegal"
        ]
        # Pré-calcula embeddings
        self.E_pos = self.model.encode(self.seeds_pos)
        self.E_neg = self.model.encode(self.seeds_neg)
        self.E_vneg = self.model.encode(self.seeds_very_neg)

    @staticmethod
    def _cosine_sim(a: np.ndarray, B: np.ndarray) -> np.ndarray:
        return B @ a

    def polarity(self, text: str, tau_pos: float = 0.05, tau_neg: float = 0.1, details: bool = False):
        e = self.model.encode([text])[0]
        sims_pos = self._cosine_sim(e, self.E_pos)
        sims_neg = self._cosine_sim(e, self.E_neg)
        sims_vneg = self._cosine_sim(e, self.E_vneg)

        pos_m = float(np.mean(sims_pos))
        neg_m = float(np.mean(sims_neg))
        vneg_m = float(np.mean(sims_vneg))

        # Heurística simples de negação
        #if any(kw in text.lower() for kw in ["not ", "never ", "no "]) and pos_m > neg_m:
            #neg_m += 0.01

        score = pos_m - vneg_m
        if score > tau_pos:
            label = "positive"
        elif score < -tau_neg:
            label = "very negative"
        else:
            label = "negative"

        if details:
            return {
                "label": label,
                "score": score,
                "pos_mean": pos_m,
                "neg_mean": neg_m,
                "very_neg_mean": vneg_m,
                "closest_positive_seed": self.seeds_pos[int(np.argmax(sims_pos))],
                "closest_negative_seed": self.seeds_neg[int(np.argmax(sims_neg))],
                "closest_very_negative_seed": self.seeds_very_neg[int(np.argmax(sims_vneg))],
            }
        return label

## Orquestração das classes

Resultado final é um DataFrame com as classes determinadas por cada modelo.

In [ ]:
# ============================
# Orquestração
# ============================
class SentimentPipeline:
    def __init__(self, cfg: Config, df):
        self.cfg = cfg
        #self.loader = DataLoader(cfg)
        self.df=df
        self.pre = TextPreprocessor()
        self.vader = VaderAnalyzer()
        self.roberta = RobertaAnalyzer(cfg)
        self.emb = EmbeddingSentiment(cfg.os_model)
    
    # Essa é a função que determina a escala, entre -1 e 1, que será utilizada para determinar a label final.
    # Esses pesos foram ajustados manualmente, com base em testes e na análise das distribuições das labels.
    @staticmethod
    def define_escala(vader, rob, emb):
        escala = 0
        match emb:
            case 'positive':
                escala += 0.4
            case 'negative':
                escala = 0.0
            case 'very negative':
                escala -= 0.1
        match rob:
            case 'positive':
                escala += 0.5
            case 'neutral':
                escala += 0.1
            case 'negative':
                escala -= 0.7
        match vader:
            case 'positive':
                escala += 0.1
            case 'neutral':
                escala += 0
            case 'negative':
                escala -= 0.05
            case 'very negative':
                escala -= 0.1       
        return escala

    def run(self) -> pd.DataFrame:
        #df = self.loader.load()
        df = self.df.copy()

        # caso queiramos rodar em uma amostra, para teste
        if self.cfg.sample and (self.cfg.sample_size > 0) and (self.cfg.sample_size < len(df)):
            df = df.sample(self.cfg.sample_size, random_state=self.cfg.random_state).reset_index(drop=True)
            logging.info("Amostra utilizada: %d linhas", len(df))

        # VADER: pré-process + score + label
        logging.info("Aplicando VADER…")
        df['vader_text'] = df['narrative'].apply(self.pre.pre_processamento_vader)
        df['vader'] = df['vader_text'].apply(self.vader.score)
        df['vader_labels'] = df['vader'].apply(self.vader.label_from_compound)

        # Texto processado simples para o modelo Roberta e OS
        df['processada'] = df['narrative'].apply(self.pre.normalize_str)

        # DistilRoBERTa
        logging.info("Aplicando DistilRoBERTa (HF pipeline)…")
        df['roberta_labels_def'] = df['processada'].apply(self.roberta.aplica_roberta)

        # Embeddings + seeds
        logging.info("Aplicando classificação por embeddings (MiniLM)…")
        df['embeddings_polarity'] = df['processada'].apply(self.emb.polarity)

        # Contagens
        logging.info("Distribuição embeddings_polarity:\n%s", df['embeddings_polarity'].value_counts().to_string())
        logging.info("Distribuição roberta_labels_def:\n%s", df['roberta_labels_def'].value_counts().to_string())
        logging.info("Distribuição vader_labels:\n%s", df['vader_labels'].value_counts().to_string())
        print('//////////////////////////////////////////////')
        print('Distribuição embeddings_polarity\n')
        print(df['embeddings_polarity'].value_counts())
        print('\n//////////////////////////////////////////////')
        print('Distribuição roberta_labels_def\n')
        print(df['roberta_labels_def'].value_counts())
        print('\n//////////////////////////////////////////////')
        print('Distribuição vader_labels:\n')
        print(df['vader_labels'].value_counts())

        #INSERE AS LABELS DEFINITIVAS
        df['escala'] = df.apply(lambda row: self.define_escala(row['vader_labels'], row['roberta_labels_def'], row['embeddings_polarity']), axis=1)

        # Exporta
        df_out = df.copy()
        df_out.to_csv(self.cfg.output_csv, index=False)
        logging.info("Arquivo salvo em: %s", self.cfg.output_csv)
        return df_out

## Aqui rodamos toda a pipeline estruturada.

Por limitações de tempo e capacidade de processamento, principalmente do modelo distilroBERTa, criamos uma estratégia de salvamento dos resultados por etapas, de 2000 inferências cada. Assim, caso o processo dê erro ou seja interropido, retemos os resultados e continuamos de onde paramos.

A base de dados utilizada tem mais de 600 mil linhas, cada uma com uma narrativa, que na maioria dos casos possui mais de 500 tokens.

In [ ]:
#Rodando toda pipeline
setup_logging()
cfg = Config()
loader = DataLoader(cfg)
base = loader.load()
lista_resultados = []
start=0
arquivos_processados = 0
for i in np.arange(2000, len(base), 2000):
    end=i
    print('Iniciando Pipeline...')
    logging.info("Configuração: %s", cfg)
    pipe = SentimentPipeline(cfg, base.loc[start:end])
    df_temp = pipe.run()
    lista_resultados.append(df_temp)
    start=end
    arquivos_processados += 1
    df_temp.to_csv(f'resultado_parcial_{arquivos_processados}.csv', index=False)
    print('Pipeline finalizada')
    print(f'Processados {end} registros. Arquivos processados: {arquivos_processados}')

df = pd.concat(lista_resultados)
df.head(3)

## Vamos incluir alguns relatos gerados por LLM para tentar equalizar as classes.

In [ ]:
cfg = Config() #alteramos o arquivo de entrada para o CSV gerado pela LLM
loader = DataLoader(cfg)
base = loader.load()
print('Iniciando Pipeline...')
logging.info("Configuração: %s", cfg)
pipe = SentimentPipeline(cfg, base)
df_temp = pipe.run()
df_temp.to_csv(f'relatos_positivos.csv', index=False)
print('Pipeline finalizada')

Iniciando Pipeline...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 125.67it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 219.19it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


//////////////////////////////////////////////
Distribuição embeddings_polarity

embeddings_polarity
negative         560
very negative    166
positive           1
Name: count, dtype: int64

//////////////////////////////////////////////
Distribuição roberta_labels_def

roberta_labels_def
positive    372
neutral     355
Name: count, dtype: int64

//////////////////////////////////////////////
Distribuição vader_labels:

vader_labels
positive         682
neutral           29
negative          11
very negative      5
Name: count, dtype: int64
Pipeline finalizada


,product,narrative,vader_text,vader,vader_labels,processada,roberta_labels_def,embeddings_polarity,escala
0,"Money transfer, virtual currency, or money ser...",This CFPB communication is submitted to formal...,This CFPB communication is submitted to formal...,0.9756,positive,this cfpb communication is submitted to formal...,positive,very negative,0.1
1,Checking or savings account,"To Whom It May Concern,\n\nI am writing to see...",To Whom It May Concern I am writing to seek as...,0.9932,positive,to whom it may concern i am writing to seek as...,positive,negative,0.5
2,Credit reporting or other personal consumer re...,This reporting agency is proactively ensuring ...,This reporting agency is proactively ensuring ...,0.9628,positive,this reporting agency is proactively ensuring ...,neutral,very negative,-0.3
3,Credit reporting or other personal consumer re...,I have provided sufficient proof under the doc...,I have provided sufficient proof under the doc...,0.8968,positive,i have provided sufficient proof under the doc...,neutral,negative,0.1
4,Credit reporting or other personal consumer re...,Subject: Formal Request for Credit Report Corr...,Subject : Formal Request for Credit Report Cor...,0.9962,positive,subject formal request for credit report corre...,neutral,negative,0.1


In [ ]:
# concatenando os resultados parciais em um único DataFrame final
df = pd.concat([df, df_temp], ignore_index=True)

## Hora de processar o DF para o modelo de ML

In [12]:
pre = TextPreprocessor()
df['processada_ml'] = df['narrative'].apply(pre.lematizador)

In [13]:
df_ml = df[['product','processada_ml', 'escala']].copy()

In [ ]:
df_ml.to_csv('df_ml.csv', index=False)